# ExpectiminimaxAgent2: debugging testbench

Simple play between Expectiminimax2 and Greedy. Used for debugging and profiling.

In [ ]:
import cProfile
import time

from nb_setup import risk_ai_game

EM2_MAX_DEPTH = 3
EM2_TOP_K = 3
RANDOM_SEED = 5678
MAX_TURNS = 500
PROFILE = True

#OPPONENT_CLS = risk_ai_game.RandomAgent
OPPONENT_CLS = risk_ai_game.GreedyAgent


class DurationTelemetry(risk_ai_game.GameTelemetry):
    def __init__(self):
        self.duration_seconds = None
        self._t0 = None

    def on_game_start(self, game_state):
        self._t0 = time.perf_counter()

    def on_before_action(self, game_state):
        pass

    def on_action(self, game_state, action, result):
        pass

    def on_game_end(self, game_state, winner):
        if self._t0 is not None:
            self.duration_seconds = time.perf_counter() - self._t0


def run_one_game(telemetry):
    if OPPONENT_CLS is risk_ai_game.RandomAgent:
        opponent = OPPONENT_CLS(0, aggression=0.5, name="P0-Random")
    else:
        opponent = OPPONENT_CLS(0, name="P0-Greedy")
    em2 = risk_ai_game.ExpectiminimaxAgent2(
        1,
        max_depth=EM2_MAX_DEPTH,
        top_k=EM2_TOP_K,
        name="P1-EM2",
    )
    agents = [opponent, em2]
    opts = risk_ai_game.RiskAIGameOptions(
        agents=agents,
        max_turns=MAX_TURNS,
        verbose=False,
        random_seed=RANDOM_SEED,
        game_telemetry=telemetry,
    )
    return risk_ai_game.run_game(opts)


telemetry = DurationTelemetry()
if PROFILE:
    pr = cProfile.Profile()
    winner = pr.runcall(run_one_game, telemetry)
    pr.print_stats(sort="cumtime")
else:
    winner = run_one_game(telemetry)

print(telemetry.duration_seconds)
print(winner)

# Expectiminimax2: Simple Performance Profile

These measurements were taken by using a modified version of the tournament notebook.

Since depth values of 6, 7, 8 currently take too long to process, it's not practical to automate them until significant performance improvments are made.

In [ ]:
import matplotlib.pyplot as plt

# this is hand captured data as when we hit depth = 8, it takes forever to run.
data = [
    {"depth": 8, "win_rate": 0.70, "sec_per_game": 3996.0},
    {"depth": 5, "win_rate": 0.60, "sec_per_game": 22.0},
    {"depth": 3, "win_rate": 0.56, "sec_per_game": 0.91},
    {"depth": 1, "win_rate": 0.525, "sec_per_game": 0.05},
]

# sort by depth
data = sorted(data, key=lambda x: x["depth"])

# unpack into lists
depths = [d["depth"] for d in data]
win_rates = [d["win_rate"] for d in data]
times = [d["sec_per_game"] for d in data]

plt.figure()
plt.plot(depths, win_rates, marker='o')
plt.xlabel("Depth")
plt.ylabel("Win Rate")
plt.title("Expectiminimax2: Win Rate vs Depth")
plt.show()

plt.figure()
plt.plot(depths, times, marker='o')
plt.yscale("log")  # important
plt.xlabel("Depth")
plt.ylabel("Seconds per Game (log scale)")
plt.title("Computation Cost vs Depth")
plt.show()

plt.figure()
plt.scatter(times, win_rates)
for d, t, w in zip(depths, times, win_rates):
    plt.text(t, w, f"d={d}", fontsize=9)
plt.xscale("log")
plt.xlabel("Seconds per Game (log scale)")
plt.ylabel("Win Rate")
plt.title("Performance vs Computation Cost")

plt.show()